In [3]:
import re
import pandas as pd
import torch
from tqdm import tqdm
from torch.nn.functional import pad
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

c:\Users\a_has\anaconda3\envs\conversation\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model_id = "gpt2"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GPT2LMHeadModel.from_pretrained(model_id).to(device)
tokenizer = GPT2TokenizerFast.from_pretrained(model_id)
start_of_sentence =" "

In [5]:
def p1(dialog, start_of_sentence):
    max_length = model.config.n_positions
    stride = 1
    
    pad_token_id = 0
    encodings = tokenizer(f"{start_of_sentence}".join(dialog), return_tensors="pt")
    seq_len = encodings.input_ids.size(1)
    padding_len = max_length -1 
    padded_input_ids = pad(torch.tensor([], dtype=torch.long), (0, padding_len), value=pad_token_id).unsqueeze(dim=0)
    encodings.input_ids = torch.cat([padded_input_ids, encodings.input_ids], dim=1)
    seq_len = encodings.input_ids.size(1)
    nlls = []
    prev_end_loc = padding_len
    for begin_loc in tqdm(range(0, seq_len, stride)):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc  # may be different from max_length on the last loop 
        begin_loc = max(padding_len, begin_loc)
        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100
        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)
            neg_log_likelihood = outputs.loss

        nlls.append(neg_log_likelihood.item())

        prev_end_loc = end_loc
        if end_loc == seq_len:
            break
    return nlls

In [6]:
el=" <SPK1> So, thanks for coming today. We're going to play a quick quiz. I’m going to ask you three questions, and I would like you to guess the three most popular answers to these questions, which were posed to a group of one hundred people. <SPK1> Ok, sounds good. <SPK1> Alright, I’m going to ask the first question: What do people normally use to transport patients in a hospital? Think about things like an ambulance, a wheelchair, or a patient's bed. What would you say, <SPK2> ? <SPK2> Mhmm… um, ambulance? <SPK1> Correct, and can you guess the next two most popular answers? <SPK2> Hmm... wheelchair? <SPK1> Yes, good! And the third one? <SPK2> Maybe... a stretcher? <SPK1> Exactly! Well done. Now, let's move on to the second question. This one is a little different: What do you think is the most popular type of vacation destination? Some of the answers we got were beach resorts, mountain retreats, and city stays. Can you guess the top three? <SPK2> Uh… beach resorts? <SPK1> Yes, that’s number one! What about number two? <SPK2> Mountain retreat? <SPK1> Exactly. And number three? <SPK2> Uh… city stays? <SPK1> Perfect! Now we have the last one: What is the most common drink people enjoy at breakfast? Can you guess? <SPK2> Coffee? <SPK1> Right! What else? <SPK2> Maybe... juice? <SPK1> Correct! And the third one? <SPK2> Milk? <SPK1> Yes, that’s exactly right! Well done! You got all the answers!"

In [14]:
pattern='<(SPK[1-9]|MOD)>'
dialog = re.sub(r'\<', r'\n<', el).split("\n")[1:]
dialog = [d.strip() for d in dialog]
display(dialog)

["<SPK1> So, thanks for coming today. We're going to play a quick quiz. I’m going to ask you three questions, and I would like you to guess the three most popular answers to these questions, which were posed to a group of one hundred people.",
 '<SPK1> Ok, sounds good.',
 "<SPK1> Alright, I’m going to ask the first question: What do people normally use to transport patients in a hospital? Think about things like an ambulance, a wheelchair, or a patient's bed. What would you say,",
 '<SPK2> ?',
 '<SPK2> Mhmm… um, ambulance?',
 '<SPK1> Correct, and can you guess the next two most popular answers?',
 '<SPK2> Hmm... wheelchair?',
 '<SPK1> Yes, good! And the third one?',
 '<SPK2> Maybe... a stretcher?',
 "<SPK1> Exactly! Well done. Now, let's move on to the second question. This one is a little different: What do you think is the most popular type of vacation destination? Some of the answers we got were beach resorts, mountain retreats, and city stays. Can you guess the top three?",
 '<SPK2

In [15]:
dialog_no_s = [re.sub(pattern,">", d) for d in dialog]
display(dialog_no_s)

["> So, thanks for coming today. We're going to play a quick quiz. I’m going to ask you three questions, and I would like you to guess the three most popular answers to these questions, which were posed to a group of one hundred people.",
 '> Ok, sounds good.',
 "> Alright, I’m going to ask the first question: What do people normally use to transport patients in a hospital? Think about things like an ambulance, a wheelchair, or a patient's bed. What would you say,",
 '> ?',
 '> Mhmm… um, ambulance?',
 '> Correct, and can you guess the next two most popular answers?',
 '> Hmm... wheelchair?',
 '> Yes, good! And the third one?',
 '> Maybe... a stretcher?',
 "> Exactly! Well done. Now, let's move on to the second question. This one is a little different: What do you think is the most popular type of vacation destination? Some of the answers we got were beach resorts, mountain retreats, and city stays. Can you guess the top three?",
 '> Uh… beach resorts?',
 '> Yes, that’s number one! What

In [16]:
perplexity_scores_p1 = p1(dialog_no_s, start_of_sentence=" ")
assert tokenizer(" ".join(dialog_no_s), return_tensors="pt").input_ids.size(1), len(perplexity_scores_p1)

 24%|██▍       | 323/1347 [03:27<10:58,  1.56it/s]
